# **1.Анализ данных**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from MyImputerMissing import MyImputerMissing
from EDA import EDA
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

from typing import Union, List, Tuple, Set, Dict

*Загрузка датасета*

**1.1 Общие сведения о датасете**

In [ ]:

try:
    df = pd.read_csv("D:\\Customer churn\\churn_synthetic_dataset.csv")
except:
    df = pd.read_csv("E:\\Customer churn\\churn_synthetic_dataset.csv")

eda = EDA()

df = df.drop(['CustomerID'], axis=1)
df[["Gender", "ContractType", "InternetService", "PaymentMethod", "Churn", "HasDependents", "PaperlessBilling"]] = df[["Gender", "ContractType", "InternetService", "PaymentMethod", "Churn", "HasDependents", "PaperlessBilling"]].astype("category")
df = df.drop_duplicates()

print(df.info())

**1.2 Пропущенные значения**

In [ ]:
nan = df.isna().sum(axis=1)
print(f'Количество записей с пропусками: {len(nan[nan > 0].index)}')

y = df["Churn"]
X = df.drop("Churn", axis=1)

imputer = MyImputerMissing()
imputer.fit(dataset = X)
X = imputer.impute()
print(X.isna().sum(axis=0))

**1.3 Распределение признаков**

In [ ]:
axes_ = eda.make_analysis_of_distribution(data = df, download = False, show_quantile = True)

**1.4 Выбросы**

In [ ]:
axes, blow = eda.make_analysis_of_blowout(data = df, method_searching = "IQR", download = False)
stats = eda.get_stats_about_blowouts(data = df).T
print(stats)

# for key, value in blow.items():
#     print(f'{key}({len(value)}):')
#     print(value)
#     print('\n')

# **2.Feature engineering**

**2.1 Матрица корреляции(Первично)**

In [ ]:
numeric = [i for i in X.columns if pd.api.types.is_numeric_dtype(df[i])]
ordinal = ["Gender", "HasDependents", "PaperlessBilling"]
onehot = [i for i in X.columns if not(i in ordinal) and not(i in numeric)]

X_ = X.drop(labels=onehot, axis = 1)
X_[ordinal] = OrdinalEncoder().fit_transform(X=X_[ordinal])

# for name in onehot:
#     encoder = OneHotEncoder(sparse_output = False)
    
#     transformed_array = encoder.fit_transform(X=X[name])
#     names = encoder.get_feature_names_out()
#     transformed_df = pd.DataFrame(data = transformed_array, columns = names, index = X_.index)
    
#     del names, transformed_array
    
#     X_ = pd.concat([X_, transformed_df], axis = 1)

print(X_)

matrix_corr = X_.corr()

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1,1,1)

im = ax.imshow(X=matrix_corr, cmap='hot')
ax.set_title("Матрица корреляции")

ax.xaxis.set_ticks(range(0,len(matrix_corr.columns)))
ax.xaxis.set_ticklabels(matrix_corr.columns)

ax.yaxis.set_ticks(range(0,len(matrix_corr.columns)))
ax.yaxis.set_ticklabels(matrix_corr.columns)

ax.tick_params(axis='x', labelrotation = 270)

fig.colorbar(im)
fig.show()

**2.2.1 Влияние признаков - Статистические методы**

In [ ]:
def get_stat_influence(dataset: pd.DataFrame, target: pd.Series) -> pd.DataFrame:
    to_ordinal = ['Gender', 'HasDependents', 'PaperlessBilling']
    to_onehot = ['ContractType', 'InternetService', 'PaymentMethod']

    onh = OneHotEncoder(sparse_output = False)

    dataset_ = dataset.drop(labels = to_onehot, axis = 1)
    dataset_[to_ordinal] = OrdinalEncoder().fit_transform(dataset_[to_ordinal])
    encoded_onh = pd.DataFrame(onh.fit_transform(dataset[to_onehot]), columns = onh.get_feature_names_out(), index = dataset.index)

    dataset_ = pd.concat([dataset_, encoded_onh], axis = 1)

    X_train, X_test, y_train, y_test = train_test_split(dataset_, target, test_size = 0.3, random_state = 47)

    best_chi2 = SelectKBest(score_func=chi2).fit(X=X_train, y=y_train)
    best_mutual = SelectKBest(score_func=mutual_info_classif).fit(X=X_train, y=y_train)

    df_chi2 = pd.DataFrame(data = best_chi2.scores_, index = dataset_.columns, columns = ["CHI2"])
    df_mutual = pd.DataFrame(data = best_mutual.scores_, index = dataset_.columns, columns = ["MUTUAL"])

    df_scores = pd.concat([df_chi2, df_mutual], axis = 1).sort_values(["CHI2"], ascending=[False])
    
    return df_scores

df_scores = get_stat_influence(dataset=X, target=y)

print(df_scores)


**2.2.2 Влияние признаков - L1(LASSO)[Не работает, т.к зависимость в данных нелинейная]**

In [ ]:
import Analyze_batches as ab
from sklearn.model_selection import GridSearchCV
from MyRegression import MyRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.dummy import DummyClassifier

def get_weight_influence(dataset:pd.DataFrame, target: pd.Series, print_info: bool = False) -> pd.Series:
    # X_ = ab.age_category(X = X[["Income", "TotalSpent", "Age", "SupportCalls"]])
    dataset["SupportCalls"] = ab.apply_lin_robust_scaler(X = dataset["SupportCalls"], method="linspace")

    param_grid = {"C" : [0.01, 0.1, 1, 10, 100], #np.logspace(-3, 3, 10)
                "threshold" : np.linspace(0.1, 1, 10),
                "solver" : ['liblinear', 'saga']}

    scoring = "roc_auc"

    lasso_model = GridSearchCV(estimator = MyRegression(penalty="l1"),
                            param_grid = param_grid,
                            cv = 5,
                            scoring = scoring,
                            refit = True)

    X_train, X_test, y_train, y_test = train_test_split(dataset, target, test_size = 0.3)

    scaler = ab.private_scaler()
    X_train = scaler.fit_transform(X=X_train, with_support_calls = False)
    X_test = scaler.transform(X=X_test)

    lasso_model.fit(X = X_train, y = y_train)

    weights = pd.Series(data = abs(lasso_model.best_estimator_.coef_.flatten()), index = X_train.columns, name = "WEIGHTS") #.sort_values(by=["weights"], axis=0, ascending=[False])

    if (print_info == True):
        print(f"Лучший C: {lasso_model.best_params_["C"]}")
        print(f"Лучший threshold: {lasso_model.best_params_["threshold"]}")
        print(f"Лучший score({scoring}): {lasso_model.best_score_}")
        print(f"accuracy: {accuracy_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"precision: {precision_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"recall: {recall_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"F1: {f1_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"ROC-AUC: {roc_auc_score(y_true=y_test, y_score=lasso_model.predict(X_test))}")

    return weights

weights = get_weight_influence(dataset=X, target=y)
print(weights)
#==========================================#
# # X_ = ab.age_category(X = X[["Income", "TotalSpent", "Age", "SupportCalls"]])
# X_ = X
# X_["SupportCalls"] = ab.apply_lin_robust_scaler(X = X_["SupportCalls"], method="linspace")
# Y_ = y

# param_grid = {"C" : [0.01, 0.1, 1, 10, 100], #np.logspace(-3, 3, 10)
#               "threshold" : np.linspace(0.1, 1, 10),
#               "solver" : ['liblinear', 'saga']}

# scoring = "roc_auc"

# lasso_model = GridSearchCV(estimator = MyRegression(penalty="l1"),
#                            param_grid = param_grid,
#                            cv = 5,
#                            scoring = scoring,
#                            refit = True)

# X_train, X_test, y_train, y_test = train_test_split(X_, Y_, test_size = 0.3)

# scaler = ab.private_scaler()
# X_train = scaler.fit_transform(X=X_train, with_support_calls = False)
# X_test = scaler.transform(X=X_test)

# lasso_model.fit(X = X_train, y = y_train)

# weights = pd.DataFrame(data = abs(lasso_model.best_estimator_.coef_.flatten()), index = X_train.columns, columns = ["weights"]).sort_values(by=["weights"], axis=0, ascending=[False])

# print(f"Лучший C: {lasso_model.best_params_["C"]}")
# print(f"Лучший threshold: {lasso_model.best_params_["threshold"]}")
# print(f"Лучший score({scoring}): {lasso_model.best_score_}")
# print(f"accuracy: {accuracy_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
# print(f"precision: {precision_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
# print(f"recall: {recall_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
# print(f"F1: {f1_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
# print(f"ROC-AUC: {roc_auc_score(y_true=y_test, y_score=lasso_model.predict(X_test))}")

# # print(weights)

# df_scores = pd.concat([df_scores, weights], axis = 1).sort_values(by=["weights"], ascending=[False])
# print(df_scores)

# # dummy = DummyClassifier(strategy='most_frequent')
# # dummy.fit(X_train, y_train)
# # y_pred_dummy = dummy.predict(X_test)

# # print("DummyClassifier (предсказывает самый частый класс):")
# # print(classification_report(y_test, y_pred_dummy))



**2.4 Отбор признаков для выборки**

In [ ]:
def get_full_influence(dataset:pd.DataFrame, target: pd.Series) -> pd.DataFrame:
    X_s = get_stat_influence(dataset=dataset, target=target)
    X_w = get_weight_influence(dataset=dataset, target=target)
    
    X = pd.concat([X_s, X_w], axis=1)
    return X

def median_selection(Features: pd.DataFrame, strict_condition: bool = False) -> Dict[str, np.ndarray]:
    #Для каждого из method возвращаем только те признаки(столбцы из таблицы), для которых выполнены условия медианной селекции
    results = dict()
    
    #method in [CHI2, WEIGHTS, MUTUAL]
    for method in Features.columns:
        median_score = Features[method].median()
        average_score = Features[method].mean()
        
        if (strict_condition == True):
            results[method] = Features.query(f"`{method}` >= @median_score and `{method}` >= @average_score")[method].index
        else:
            results[method] = Features.query(f"`{method}` >= @median_score or `{method}` >= @average_score")[method].index
    
    return results

def vote_selection(Features: pd.DataFrame) -> np.ndarray:
    #Возвращаем только те признаки(столбцы из таблицы), за которые проголосовали 70% method
    def solve_params() -> Dict[str, np.ndarray]:
        params = dict()
        
        for method in Features.columns:
            params[method] = [Features[method].mean(), Features[method].median()]
            
        return params
    
    def vote(feature: Union[float, int], mean_median: Union[List, np.ndarray]) -> bool:
        return feature >= mean_median[0] or feature >= mean_median[1]
    
    finally_features = list()
    params = solve_params()
    max_len_votes = len(Features.columns)
    
    #method in [CHI2, WEIGHTS, MUTUAL]
    for feature in Features.index:
        votes = list()
        for method in Features.columns:
            votes.append(vote(Features[method][feature], params[method]))
            
        if sum(votes) >= round(max_len_votes*0.7):
            finally_features.append(feature)
            
    return finally_features

def compare_table(Scores: pd.DataFrame) -> pd.DataFrame:
    proposal = {"Vote selection" : vote_selection(Features = Scores),
                "Median selection" : median_selection(Features = Scores),
                "Median selection(strict)" : median_selection(Features = Scores, strict_condition = True)}
    
    table = pd.DataFrame(columns = list(proposal.keys()), index = Scores.index)
    
    #Для каждого способа(way) смотрим выбранные им признаки
    #way in [VOTE, MEDIAN, MEDIAN_STRICT]
    #method in [CHI2, WEIGHTS, MUTUAL]
    for way in proposal.keys():
        for feature in table.index:
            if table[way][feature] is np.nan:
                    table[way][feature] = 0
                    
            if way == "Vote selection":
                if feature in proposal[way]:
                    table[way][feature] += 1
                else:
                    continue
            else:
                for method in proposal[way].keys():
                    if feature in proposal[way][method]:
                        table[way][feature] += 1
                    else:
                        continue
    return table
             
def dump_table_excel(table: pd.DataFrame, path: str = None):
    import os
    if path is None:
        path = f'{os.getcwd()}\\Compare_table.xlsx'
        table.to_excel(path, sheet_name="Report")
    else:
        table.to_excel(path, sheet_name="Report")
    print(f'Сохранено в [{path}]')

df_scores = get_full_influence(dataset=X, target=y)
      
table = compare_table(Scores=df_scores)
dump_table_excel(table=table)    

print(len(median_selection(df_scores, strict_condition=True)), len(df_scores.index))
print(len(vote_selection(df_scores)), len(df_scores.index))

# **3.Pipeline model**

**3.1 Решающие деревья**

In [ ]:
import pandas as pd
import numpy as np
from typing import Union, List, Tuple, Set, Dict
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer


def get_stat_influence(dataset: pd.DataFrame, target: pd.Series) -> pd.DataFrame:
    from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
    from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
    from sklearn.model_selection import train_test_split

    to_ordinal = ['Gender', 'HasDependents', 'PaperlessBilling']
    to_onehot = ['ContractType', 'InternetService', 'PaymentMethod']

    onh = OneHotEncoder(sparse_output = False)

    dataset_ = dataset.drop(labels = to_onehot, axis = 1)
    dataset_[to_ordinal] = OrdinalEncoder().fit_transform(dataset_[to_ordinal])
    encoded_onh = pd.DataFrame(onh.fit_transform(dataset[to_onehot]), columns = onh.get_feature_names_out(), index = dataset.index)

    dataset_ = pd.concat([dataset_, encoded_onh], axis = 1)

    X_train, X_test, y_train, y_test = train_test_split(dataset_, target, test_size = 0.3, random_state = 47)

    best_chi2 = SelectKBest(score_func=chi2).fit(X=X_train, y=y_train)
    best_mutual = SelectKBest(score_func=mutual_info_classif).fit(X=X_train, y=y_train)

    df_chi2 = pd.DataFrame(data = best_chi2.scores_, index = dataset_.columns, columns = ["CHI2"])
    df_mutual = pd.DataFrame(data = best_mutual.scores_, index = dataset_.columns, columns = ["MUTUAL"])

    df_scores = pd.concat([df_chi2, df_mutual], axis = 1).sort_values(["CHI2"], ascending=[False])
    
    return df_scores

def get_weight_influence(dataset:pd.DataFrame, target: pd.Series, data_is_scaled: bool = False, print_info: bool = False) -> pd.Series:
    import Analyze_batches as ab
    from sklearn.model_selection import GridSearchCV
    from MyRegression import MyRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    param_grid = {"C" : [0.01, 0.1, 1, 10, 100], #np.logspace(-3, 3, 10)
                "threshold" : np.linspace(0.1, 1, 10),
                "solver" : ['liblinear', 'saga']}

    scoring = "roc_auc"

    lasso_model = GridSearchCV(estimator = MyRegression(penalty="l1"),
                            param_grid = param_grid,
                            cv = 5,
                            scoring = scoring,
                            refit = True)

    if (data_is_scaled == False):    
        dataset["SupportCalls"] = ab.apply_lin_robust_scaler(X = dataset["SupportCalls"], method="linspace")
    
    X_train, X_test, y_train, y_test = train_test_split(dataset, target, test_size = 0.3)

    if (data_is_scaled == False):    
        scaler = ab.private_scaler()
        X_train = scaler.fit_transform(X=X_train, with_support_calls = False)
        X_test = scaler.transform(X=X_test)

    lasso_model.fit(X = X_train, y = y_train)

    weights = pd.Series(data = abs(lasso_model.best_estimator_.coef_.flatten()), index = X_train.columns, name = "WEIGHTS") #.sort_values(by=["weights"], axis=0, ascending=[False])

    if (print_info == True):
        print(f"Лучший C: {lasso_model.best_params_["C"]}")
        print(f"Лучший threshold: {lasso_model.best_params_["threshold"]}")
        print(f"Лучший score({scoring}): {lasso_model.best_score_}")
        print(f"accuracy: {accuracy_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"precision: {precision_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"recall: {recall_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"F1: {f1_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"ROC-AUC: {roc_auc_score(y_true=y_test, y_score=lasso_model.predict(X_test))}")

    return weights

def get_full_influence(dataset:pd.DataFrame, target: pd.Series, data_is_scaled: bool = False) -> pd.DataFrame:
    inf_s = get_stat_influence(dataset=dataset, target=target)
    inf_w = get_weight_influence(dataset=dataset, target=target, data_is_scaled=data_is_scaled)
    
    inf_full = pd.concat([inf_s, inf_w], axis=1)
    return inf_full

def vote_selection(Features_impact: pd.DataFrame) -> np.ndarray:
    #Возвращаем только те признаки(столбцы из таблицы), за которые проголосовали 70% method
    def solve_params() -> Dict[str, np.ndarray]:
        params = dict()
        
        for method in Features_impact.columns:
            params[method] = [Features_impact[method].mean(), Features_impact[method].median()]
            
        return params
    
    def vote(feature: Union[float, int], mean_median: Union[List, np.ndarray]) -> bool:
        return feature >= mean_median[0] or feature >= mean_median[1]
    
    finally_features = list()
    params = solve_params()
    max_len_votes = len(Features_impact.columns)
    
    #method in [CHI2, WEIGHTS, MUTUAL]
    for feature in Features_impact.index:
        votes = list()
        for method in Features_impact.columns:
            votes.append(vote(Features_impact[method][feature], params[method]))
            
        if sum(votes) >= round(max_len_votes*0.7):
            finally_features.append(feature)
            
    return finally_features

def impute_missing(X: pd.DataFrame) -> pd.DataFrame:
    from MyImputerMissing import MyImputerMissing
    
    nones = X.isna().sum(axis=0)
    
    if (len(nones[nones > 0].index) > 0):
        del nones
        imputer = MyImputerMissing()
        imputer.fit(dataset = X)
        X = imputer.impute()
    
    return X

def load_dataset() -> Tuple[pd.DataFrame, pd.Series]:
    try:
        df = pd.read_csv("D:\\Customer churn\\churn_synthetic_dataset.csv") #ПК
    except:
        df = pd.read_csv("E:\\Customer churn\\churn_synthetic_dataset.csv") #Ноутбук

    df = df.drop(['CustomerID'], axis=1)
    df[["Gender", "ContractType", "InternetService", "PaymentMethod", "Churn", "HasDependents", "PaperlessBilling"]] = df[["Gender", "ContractType", "InternetService", "PaymentMethod", "Churn", "HasDependents", "PaperlessBilling"]].astype("category")
    df = df.drop_duplicates()
    
    y = df["Churn"]
    X = df.drop("Churn", axis=1)
    
    X = impute_missing(X = X)
    
    return X, y

def create_batch(X: pd.DataFrame, y: pd.Series) -> Tuple[pd.DataFrame, pd.Series]:
    import Analyze_batches as ab
    from sklearn.model_selection import train_test_split
    
    X = impute_missing(X = X)
    
    X["SupportCalls"] = ab.apply_lin_robust_scaler(X = X["SupportCalls"], method="linspace")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)
    
    scaler = ab.private_scaler()
    X_train = scaler.fit_transform(X=X_train, with_support_calls = False)
    X_test = scaler.transform(X=X_test)
    
    X_ = pd.concat([X_train, X_test], axis=0)
    y_ = pd.concat([y_train, y_test], axis=0)
    
    influence = get_full_influence(dataset=X, target=y)
    voted_features = vote_selection(Features_impact=influence)
    
    return (X_[voted_features], y_)

def train_dummy(print_info: bool = False):
    from sklearn.model_selection import train_test_split
    from sklearn.dummy import DummyClassifier
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score 

    X, y = load_dataset()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)
    
    model = DummyClassifier(strategy='most_frequent')
    model.fit(X = X_train, y = y_train)
    
    if (print_info == True):      
        print(f"accuracy: {round(accuracy_score(y_true=y_test, y_pred=model.predict(X_test)), 3)}")
        print(f"precision: {round(precision_score(y_true=y_test, y_pred=model.predict(X_test)), 3)}")
        print(f"recall: {round(recall_score(y_true=y_test, y_pred=model.predict(X_test)), 3)}")
        print(f"F1: {round(f1_score(y_true=y_test, y_pred=model.predict(X_test)), 3)}")
        print(f"ROC-AUC: {round(roc_auc_score(y_true=y_test, y_score=model.predict(X_test)), 3)}")
    
def train_tree(print_info: bool = False, first_batch : bool = True) -> Tuple[DecisionTreeClassifier, pd.DataFrame]:
    from sklearn.model_selection import train_test_split
    from sklearn.model_selection import GridSearchCV
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score     
    from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder


    def full_original_batch() -> Tuple[pd.DataFrame, pd.Series]:
        X, y = load_dataset()
    
        to_ordinal = ['Gender', 'HasDependents', 'PaperlessBilling']
        to_onehot = ['ContractType', 'InternetService', 'PaymentMethod']

        onh = OneHotEncoder(sparse_output = False)

        X_ = X.drop(labels = to_onehot, axis = 1)
        X_[to_ordinal] = OrdinalEncoder().fit_transform(X_[to_ordinal])
        encoded_onh = pd.DataFrame(onh.fit_transform(X[to_onehot]), columns = onh.get_feature_names_out(), index = X.index)
        X_ = pd.concat([X_, encoded_onh], axis = 1)
        
        return (X_, y)    
    
    batches = {"Original" : full_original_batch(), "Preprocessing" : create_batch(*load_dataset())}
    
    param_grid = {"criterion": ['gini', 'entropy', 'log_loss'],
                    "max_depth": [1, 5, 9],
                    "min_samples_leaf": [5, 10, 15],
                    "min_samples_split": [15, 30, 45]} #min_samples_split должен быть хотя бы в 3 раза больше min_samples_leaf, чтобы были созданы нормальные leaf
        
    grid = GridSearchCV(estimator=DecisionTreeClassifier(), param_grid=param_grid)
    
    if (print_info == True):
        for name, batch in batches.items():
            X_train, X_test, y_train, y_test = train_test_split(batch[0], batch[1], test_size = 0.3)
            
            grid.fit(X=X_train, y=y_train)
            
            print(f"========={name}=========")
            print(f"accuracy: {round(accuracy_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
            print(f"precision: {round(precision_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
            print(f"recall: {round(recall_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
            print(f"F1: {round(f1_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
            print(f"ROC-AUC: {round(roc_auc_score(y_true=y_test, y_score=grid.predict(X_test)), 3)}")
            
            print(grid.best_params_)
            print(f"========={len(name)*'='}=========")
            
            batch_ = batch
    else:
        if first_batch == True:
            batch_ = batches["Original"]
        else:
            batch_ = batches["Preprocessing"]
            
        X_train, X_test, y_train, y_test = train_test_split(batch_[0], batch_[1], test_size = 0.3)
        
        grid.fit(X=X_train, y=y_train)
    

    return (grid.best_estimator_, batch_[0])

def shap_analysis(tree: DecisionTreeClassifier, X_dataset: pd.DataFrame):
    import shap
    
    explainer1 = shap.TreeExplainer(tree)
    # explainer2 = shap.KernelExplainer(tree)
    
    shap_values1 = explainer1.shap_values(X_dataset)
    # shap_values2 = explainer2.shap_values(X_dataset)
    
    shap.summary_plot(shap_values1[:, :, 1], X_dataset)

def get_full_influence_pipe() -> pd.DataFrame:
    dataset, target = load_dataset()
    
    inf_s = get_stat_influence(dataset=dataset, target=target)
    inf_w = get_weight_influence(dataset=dataset, target=target, data_is_scaled=False)
    
    inf_full = pd.concat([inf_s, inf_w], axis=1)
    return inf_full

def create_batch_pipe(X:pd.DataFrame) -> pd.DataFrame:
    import Analyze_batches as ab
    
    X["SupportCalls"] = ab.apply_lin_robust_scaler(X = X["SupportCalls"], method="linspace")
    
    scaler = ab.private_scaler()
    X_ = scaler.fit_transform(X=X, with_support_calls = False)
    
    influence = get_full_influence_pipe()
    # voted_features = vote_selection(Features_impact=influence)

    return X_
 
def pipeline_tree() -> DecisionTreeClassifier:
    from sklearn.model_selection import GridSearchCV
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score 
    from sklearn.model_selection import train_test_split
    
    imputing = FunctionTransformer(impute_missing, validate=False)
    batching = FunctionTransformer(create_batch_pipe, validate=False)

    tree = Pipeline(steps=[("imputer", imputing),
                            ("preprocessing", batching),
                            ("model", DecisionTreeClassifier())])
    
    param_grid = {"model__criterion": ['gini', 'entropy', 'log_loss'],
                    "model__max_depth": [1, 5, 9],
                    "model__min_samples_leaf": [5, 10, 15],
                    "model__min_samples_split": [15, 30, 45]}
    grid = tree # GridSearchCV(estimator=tree, param_grid=param_grid)
    
    X, y = load_dataset()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)
    
    grid.fit(X=X_train, y=y_train)
    
    print(f"accuracy: {round(accuracy_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
    print(f"precision: {round(precision_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
    print(f"recall: {round(recall_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
    print(f"F1: {round(f1_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
    print(f"ROC-AUC: {round(roc_auc_score(y_true=y_test, y_score=grid.predict(X_test)), 3)}")
    
    # print(grid.best_params_)

    return grid

def graph_tree(tree: DecisionTreeClassifier):
    from sklearn.tree import plot_tree
    # tree, X = train_tree(print_info=True)
    
    plot_tree(tree)
    

print(f"////////Tree////////")
tree, X = train_tree(print_info=True)
# print(f"////////Dummy////////")
# train_dummy(print_info=True)
# shap_analysis(tree=tree, X_dataset=X)

# print(f"////////Pipeline_Tree////////")
# tree_ = pipeline_tree()

graph_tree(tree)
print(X.columns)



**3.2 Ближайшие соседи**

In [ ]:
import pandas as pd
import numpy as np
from typing import Union, List, Tuple, Set, Dict
from sklearn.preprocessing import FunctionTransformer
from sklearn.neighbors import KNeighborsClassifier

def get_stat_influence(dataset: pd.DataFrame, target: pd.Series) -> pd.DataFrame:
    from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
    from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
    from sklearn.model_selection import train_test_split

    to_ordinal = ['Gender', 'HasDependents', 'PaperlessBilling']
    to_onehot = ['ContractType', 'InternetService', 'PaymentMethod']

    onh = OneHotEncoder(sparse_output = False)

    dataset_ = dataset.drop(labels = to_onehot, axis = 1)
    dataset_[to_ordinal] = OrdinalEncoder().fit_transform(dataset_[to_ordinal])
    encoded_onh = pd.DataFrame(onh.fit_transform(dataset[to_onehot]), columns = onh.get_feature_names_out(), index = dataset.index)

    dataset_ = pd.concat([dataset_, encoded_onh], axis = 1)

    X_train, X_test, y_train, y_test = train_test_split(dataset_, target, test_size = 0.3, random_state = 47)

    best_chi2 = SelectKBest(score_func=chi2).fit(X=X_train, y=y_train)
    best_mutual = SelectKBest(score_func=mutual_info_classif).fit(X=X_train, y=y_train)

    df_chi2 = pd.DataFrame(data = best_chi2.scores_, index = dataset_.columns, columns = ["CHI2"])
    df_mutual = pd.DataFrame(data = best_mutual.scores_, index = dataset_.columns, columns = ["MUTUAL"])

    df_scores = pd.concat([df_chi2, df_mutual], axis = 1).sort_values(["CHI2"], ascending=[False])
    
    return df_scores

def get_weight_influence(dataset:pd.DataFrame, target: pd.Series, data_is_scaled: bool = False, print_info: bool = False) -> pd.Series:
    import Analyze_batches as ab
    from sklearn.model_selection import GridSearchCV
    from MyRegression import MyRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    param_grid = {"C" : [0.01, 0.1, 1, 10, 100], #np.logspace(-3, 3, 10)
                "threshold" : np.linspace(0.1, 1, 10),
                "solver" : ['liblinear', 'saga']}

    scoring = "roc_auc"

    lasso_model = GridSearchCV(estimator = MyRegression(penalty="l1"),
                            param_grid = param_grid,
                            cv = 5,
                            scoring = scoring,
                            refit = True)

    if (data_is_scaled == False):    
        dataset["SupportCalls"] = ab.apply_lin_robust_scaler(X = dataset["SupportCalls"], method="linspace")
    
    X_train, X_test, y_train, y_test = train_test_split(dataset, target, test_size = 0.3)

    if (data_is_scaled == False):    
        scaler = ab.private_scaler()
        X_train = scaler.fit_transform(X=X_train, with_support_calls = False)
        X_test = scaler.transform(X=X_test)

    lasso_model.fit(X = X_train, y = y_train)

    weights = pd.Series(data = abs(lasso_model.best_estimator_.coef_.flatten()), index = X_train.columns, name = "WEIGHTS") #.sort_values(by=["weights"], axis=0, ascending=[False])

    if (print_info == True):
        print(f"Лучший C: {lasso_model.best_params_["C"]}")
        print(f"Лучший threshold: {lasso_model.best_params_["threshold"]}")
        print(f"Лучший score({scoring}): {lasso_model.best_score_}")
        print(f"accuracy: {accuracy_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"precision: {precision_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"recall: {recall_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"F1: {f1_score(y_true=y_test, y_pred=lasso_model.predict(X_test))}")
        print(f"ROC-AUC: {roc_auc_score(y_true=y_test, y_score=lasso_model.predict(X_test))}")

    return weights

def get_full_influence(dataset:pd.DataFrame, target: pd.Series, data_is_scaled: bool = False) -> pd.DataFrame:
    inf_s = get_stat_influence(dataset=dataset, target=target)
    inf_w = get_weight_influence(dataset=dataset, target=target, data_is_scaled=data_is_scaled)
    
    inf_full = pd.concat([inf_s, inf_w], axis=1)
    return inf_full

def vote_selection(Features_impact: pd.DataFrame) -> np.ndarray:
    #Возвращаем только те признаки(столбцы из таблицы), за которые проголосовали 70% method
    def solve_params() -> Dict[str, np.ndarray]:
        params = dict()
        
        for method in Features_impact.columns:
            params[method] = [Features_impact[method].mean(), Features_impact[method].median()]
            
        return params
    
    def vote(feature: Union[float, int], mean_median: Union[List, np.ndarray]) -> bool:
        return feature >= mean_median[0] or feature >= mean_median[1]
    
    finally_features = list()
    params = solve_params()
    max_len_votes = len(Features_impact.columns)
    
    #method in [CHI2, WEIGHTS, MUTUAL]
    for feature in Features_impact.index:
        votes = list()
        for method in Features_impact.columns:
            votes.append(vote(Features_impact[method][feature], params[method]))
            
        if sum(votes) >= round(max_len_votes*0.7):
            finally_features.append(feature)
            
    return finally_features

def impute_missing(X: pd.DataFrame) -> pd.DataFrame:
    from MyImputerMissing import MyImputerMissing
    
    nones = X.isna().sum(axis=0)
    
    if (len(nones[nones > 0].index) > 0):
        del nones
        imputer = MyImputerMissing()
        imputer.fit(dataset = X)
        X = imputer.impute()
    
    return X

def load_dataset() -> Tuple[pd.DataFrame, pd.Series]:
    try:
        df = pd.read_csv("D:\\Customer churn\\churn_synthetic_dataset.csv") #ПК
    except:
        df = pd.read_csv("E:\\Customer churn\\churn_synthetic_dataset.csv") #Ноутбук

    df = df.drop(['CustomerID'], axis=1)
    df[["Gender", "ContractType", "InternetService", "PaymentMethod", "Churn", "HasDependents", "PaperlessBilling"]] = df[["Gender", "ContractType", "InternetService", "PaymentMethod", "Churn", "HasDependents", "PaperlessBilling"]].astype("category")
    df = df.drop_duplicates()
    
    y = df["Churn"]
    X = df.drop("Churn", axis=1)
    
    X = impute_missing(X = X)
    
    return X, y

def create_batch(X: pd.DataFrame, y: pd.Series) -> Tuple[pd.DataFrame, pd.Series]:
    import Analyze_batches as ab
    from sklearn.model_selection import train_test_split
    
    X = impute_missing(X = X)
    
    X["SupportCalls"] = ab.apply_lin_robust_scaler(X = X["SupportCalls"], method="linspace")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)
    
    scaler = ab.private_scaler()
    X_train = scaler.fit_transform(X=X_train, with_support_calls = False)
    X_test = scaler.transform(X=X_test)
    
    X_ = pd.concat([X_train, X_test], axis=0)
    y_ = pd.concat([y_train, y_test], axis=0)
    
    influence = get_full_influence(dataset=X, target=y)
    voted_features = vote_selection(Features_impact=influence)
    
    return (X_[voted_features], y_)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score     
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder


def train_knn(print_info: bool = False, first_batch : bool = True) -> Tuple[KNeighborsClassifier, pd.DataFrame]:

    def full_original_batch() -> Tuple[pd.DataFrame, pd.Series]:
        X, y = load_dataset()
    
        to_ordinal = ['Gender', 'HasDependents', 'PaperlessBilling']
        to_onehot = ['ContractType', 'InternetService', 'PaymentMethod']

        onh = OneHotEncoder(sparse_output = False)

        X_ = X.drop(labels = to_onehot, axis = 1)
        X_[to_ordinal] = OrdinalEncoder().fit_transform(X_[to_ordinal])
        encoded_onh = pd.DataFrame(onh.fit_transform(X[to_onehot]), columns = onh.get_feature_names_out(), index = X.index)
        X_ = pd.concat([X_, encoded_onh], axis = 1)
        
        return (X_, y)    

    batches = {"Original" : full_original_batch(), "Preprocessing" : create_batch(*load_dataset())}
    
    param_grid = {
                "n_neighbors": [3, 5, 7, 13],
                "algorithm": ["kd_tree", "ball_tree"],
                "weights": ['uniform', "distance"],
                "metric": ['minkowski', 'cosine', 'euclidean', 'jaccard']} 
        
    grid = GridSearchCV(estimator=KNeighborsClassifier(), param_grid=param_grid)
    
    if (print_info == True):
        for name, batch in batches.items():
            X_train, X_test, y_train, y_test = train_test_split(batch[0], batch[1], test_size = 0.3)
            
            grid.fit(X=X_train, y=y_train)
            
            print(f"========={name}=========")
            print(f"accuracy: {round(accuracy_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
            print(f"precision: {round(precision_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
            print(f"recall: {round(recall_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
            print(f"F1: {round(f1_score(y_true=y_test, y_pred=grid.predict(X_test)), 3)}")
            print(f"ROC-AUC: {round(roc_auc_score(y_true=y_test, y_score=grid.predict(X_test)), 3)}")
            
            print(grid.best_params_)
            print(f"========={len(name)*'='}=========")
            
            batch_ = batch
    else:
        if first_batch == True:
            batch_ = batches["Original"]
        else:
            batch_ = batches["Preprocessing"]
            
        X_train, X_test, y_train, y_test = train_test_split(batch_[0], batch_[1], test_size = 0.3)
        
        grid.fit(X=X_train, y=y_train)
    

    return (grid.best_estimator_, batch_[0])

train_knn(print_info = True)